In [1]:
%who

Interactive namespace is empty.


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

In [3]:
# Loading Data
df = pd.read_csv(r"C:\Users\user\Documents\00. Github Projects\01. Project 1 Credit-Risk Preditction\02. Clean Data\clean_loans_final.csv")

In [4]:
# Feature Engineering
df['fico_avg'] = (df['fico_range_low'] + df['fico_range_high']) / 2
df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)
df['installment_to_income'] = df['installment'] / (df['annual_inc'] / 12 + 1)
df['credit_utilization'] = df['revol_bal'] / (df['total_acc'] + 1)
df['rate_to_dti'] = df['int_rate'] / (df['dti'] + 0.1)
df['delinquency_rate'] = df['delinq_2yrs'] / (df['total_acc'] + 1)
df['public_record_rate'] = df['pub_rec'] / (df['total_acc'] + 1)
df['mortgage_rate'] = df['mort_acc'] / (df['total_acc'] + 1)

In [5]:
# Removing Unnecessary Columns
df = df.drop(columns=['earliest_cr_line'], errors='ignore')
print(f"Dropped 'earliest_cr_line'")
print(f"Columns before encoding: {df.shape[1]}")

# One-Hot Encoding
cat_cols = ['grade', 'sub_grade', 'home_ownership', 'verification_status', 'purpose']
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)
print(f"One-hot encoding complete. Now have {df_encoded.shape[1]} columns")

# Split by Date
df_encoded['issue_d'] = pd.to_datetime(df_encoded['issue_d'], format='%b-%Y')
df_encoded = df_encoded.sort_values('issue_d')

y = df_encoded['is_bad']
X = df_encoded.drop(columns=['is_bad', 'issue_d'])

split_idx = int(len(X) * 0.8)
X_train = X.iloc[:split_idx].copy()
X_test = X.iloc[split_idx:].copy()
y_train = y.iloc[:split_idx].copy()
y_test = y.iloc[split_idx:].copy()

print(f"Train: {X_train.shape[0]} loans | Test: {X_test.shape[0]} loans")
print(f"   Train default rate: {y_train.mean()*100:.2f}%")
print(f"   Test default rate:  {y_test.mean()*100:.2f}%")

# Fixing any Remaining Text Columns
# Convert 'pymnt_plan' if it exists
if 'pymnt_plan' in X_train.columns:
    X_train['pymnt_plan'] = X_train['pymnt_plan'].map({'n': 0, 'y': 1}).fillna(0)
    X_test['pymnt_plan'] = X_test['pymnt_plan'].map({'n': 0, 'y': 1}).fillna(0)

# Convert any other object columns to numeric
for col in X_train.select_dtypes(include=['object']).columns:
    print(f"   Converting '{col}' to numeric...")
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce').fillna(0)
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce').fillna(0)

print("All text columns converted to numbers")

Dropped 'earliest_cr_line'
Columns before encoding: 38
One-hot encoding complete. Now have 93 columns
Train: 1099108 loans | Test: 274777 loans
   Train default rate: 19.46%
   Test default rate:  20.18%
   Converting 'term' to numeric...
All text columns converted to numbers


In [6]:
# Scale Numeric Features
continuous_cols = [
    'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'term', 'int_rate',
    'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'inq_last_6mths',
    'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'mort_acc', 'fic	o_avg', 'loan_to_income', 'installment_to_income',
    'credit_utilization', 'rate_to_dti', 'delinquency_rate',
    'public_record_rate', 'mortgage_rate'
]

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

# Only scale columns that exist
scale_cols = [col for col in continuous_cols if col in X_train.columns]
X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

print(f"Scaling complete. Training shape: {X_train_scaled.shape}")

Scaling complete. Training shape: (1099108, 91)


In [7]:
# Drop identifier columns 
columns_to_drop = ['id', 'member_id', 'funded_amnt', 'funded_amnt_inv', 'pymnt_plan']
# 'funded_amnt' and 'funded_amnt_inv' are nearly identical to 'loan_amnt' – we don't need them
# 'pymnt_plan' is already 0/1 but we can drop it for simplicity

X_train_scaled = X_train_scaled.drop(columns=columns_to_drop, errors='ignore')
X_test_scaled = X_test_scaled.drop(columns=columns_to_drop, errors='ignore')

print(f"✅ Dropped useless columns. Training shape: {X_train_scaled.shape}")

✅ Dropped useless columns. Training shape: (1099108, 87)


In [8]:
from sklearn.metrics import roc_auc_score, classification_report

print("\n" + "="*50)
print("📊 RE-TRAINING LOGISTIC REGRESSION (WITHOUT ID COLUMNS)")
print("="*50)

log_reg = LogisticRegression(max_iter=1000, solver='liblinear', random_state=42,C=0.1, class_weight='balanced')
log_reg.fit(X_train_scaled, y_train)
log_reg.fit(X_train_scaled, y_train)


y_pred = log_reg.predict(X_test_scaled)
y_proba = log_reg.predict_proba(X_test_scaled)[:, 1]


auc = roc_auc_score(y_test, y_proba)
print(f"\n✅ AUC-ROC: {auc:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))


📊 RE-TRAINING LOGISTIC REGRESSION (WITHOUT ID COLUMNS)
Line 7 Done
Line 9 Done
Line 11 Done
Line 14 Done
Line 16 Done
Line 19 Done

✅ AUC-ROC: 0.6828

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.59      0.71    219323
           1       0.30      0.67      0.41     55454

    accuracy                           0.61    274777
   macro avg       0.59      0.63      0.56    274777
weighted avg       0.76      0.61      0.65    274777

